# Match metadata: runtime and operations notes

Operator guide for local execution, tuning, and recovery.


## Core pipeline sequence (full run)

Run from repository root with `$PDIR` set (e.g. `/data/scratch/philip.brohan/ADRQ`).

```bash
# 1) RR DATA ingest (combined station CSVs under Rainfall-Rescue/DATA)
python scripts/build_rainfall_rescue_parquet.py

# 2) RR ALLSHEETS ingest (individual source-sheet CSVs under Rainfall-Rescue/ALLSHEETS)
python scripts/build_allsheets_parquet.py

# 3) Daily transcriptions ingest
scripts/slurm/submit_ensemble_ingest.sh

# 4) Daily transcriptions QC
scripts/slurm/submit_transcription_qc.sh

# 5) Match metadata (submit in three stages; wait for each stage to finish OK)
scripts/slurm/submit_all.sh
# wait for rqc_merge to complete successfully before continuing
sbatch --export=ALL,RQC_SLURM_DIR="$PWD/scripts/slurm" scripts/slurm/assign_metadata.sbatch
# wait for rqc_assign_meta to complete successfully before continuing
scripts/slurm/submit_allsheets.sh

# 6) QC monthly total (QC1)
scripts/slurm/submit_qc.sh

# 7) QC regional stats (QC2 stage 1)
scripts/slurm/submit_daily_consensus.sh
REGIONAL_MEM_MB=32000 REGIONAL_TIME_MIN=180 REGIONAL_NUM_SHARDS=400 scripts/slurm/submit_regional_stats.sh

# 8) QC secondary (QC2 stage 2 model train + score)
scripts/slurm/submit_secondary_qc.sh

# 9) Export SEF
scripts/slurm/submit_sef_export.sh

# 9) Build database from SEF for fast checks.
scripts/slurm/submit_sef_analysis.sh
```

Notes:
- RR DATA and RR ALLSHEETS ingests are separate steps.
- match_metadata requires the ALLSHEETS parquet dataset to exist.
- Do not chain the three Step 5 commands with `&&`; SLURM submissions are asynchronous.
- `assign_metadata.sbatch` must be submitted with `RQC_SLURM_DIR` exported as shown above.

## Monitoring

```bash
ls -t $PDIR/slurm_logs | head
cat $PDIR/monthly_similarity_parquet/run_manifest/current.json
python scripts/local/verify_match_metadata_manifest.py --comparison-root "$PDIR/monthly_similarity_parquet"
```

Check shard directories and session outputs if a stage fails before rerunning.


## Single-command wrapper (optional)

If you want one command for the metadata-matching stages, use the local wrapper:

```bash
scripts/local/submit_local.sh match_metadata
```

For full SLURM production runs, keep Step 5 staged (submit similarity, wait; submit assign metadata with `RQC_SLURM_DIR`, wait; then submit ALLSHEETS).

The shell `&&` operator only checks submission success, not job completion, so chaining those SLURM submissions can fail due to unmet dependencies.

## Tuning knobs

Primary local knobs are configured in scripts/slurm/config.sh and scripts/slurm/config.sh:

- SLURM_QOS
- LOCAL_TOTAL_CORES
- LOCAL_TOTAL_MEM_MB
- NUM_SHARDS and ALLSHEETS_NUM_SHARDS
- TOP_K, MIN_OVERLAP, UNCERTAINTY_WEIGHT, BATCH_SIZE
